In [44]:
from census import Census
from pygris import block_groups
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [45]:
# =====================================================
# CONFIG
# =====================================================

API_KEY = open("../../data/census/api.txt").read().strip()

STATE_FIPS = "06"
COUNTY_FIPS = "001"

In [46]:
# =====================================================
# ACS DATA
# =====================================================

c = Census(API_KEY, year=2024)

# Define your variables: code -> friendly column name
ACS_VARIABLES = {
    "B01003_001E": "population",
    "B19013_001E": "median_household_income",
    "B25077_001E": "median_home_value",
    # Add more as needed...
}

print("Downloading ACS data...")

acs = c.acs5.state_county_blockgroup(
    tuple(ACS_VARIABLES.keys()),
    STATE_FIPS,
    COUNTY_FIPS,
    Census.ALL
)

df = pd.DataFrame(acs)

# Build GEOID
df["tract"] = df["tract"].str.zfill(6)
df["block group"] = df["block group"].str.zfill(1)
df["GEOID"] = (
    df["state"]
    + df["county"]
    + df["tract"]
    + df["block group"]
)

# Convert each variable to numeric and rename
for code, col_name in ACS_VARIABLES.items():
    df[col_name] = pd.to_numeric(df[code], errors="coerce")

# Drop raw census code columns, keep only friendly names + GEOID
raw_cols = list(ACS_VARIABLES.keys()) + ["state", "county", "tract", "block group"]
df = df.drop(columns=raw_cols)

# =====================================================
# BLOCK GROUP GEOMETRIES
# =====================================================

print("Downloading block group geometries...")

bg = block_groups(
    state="CA",
    county="Alameda",
    year=2024
)

# =====================================================
# JOIN ACS DATA
# =====================================================

acs_cols = ["GEOID"] + list(ACS_VARIABLES.values())

gdf = bg.merge(
    df[acs_cols],
    on="GEOID",
    how="left"
)

cols_to_drop = [
    "STATEFP", "COUNTYFP", "TRACTCE", "BLKGRPCE",
    "GEOIDFQ", "NAMELSAD", "MTFCC", "FUNCSTAT",
    "ALAND", "AWATER", "INTPTLON", "INTPTLAT"
]

gdf = gdf.drop(columns=cols_to_drop)

gdf.head()

Using FIPS code '06' for input 'CA'
Using FIPS code '001' for input 'Alameda'


,GEOID,geometry,population,median_household_income,median_home_value
0,060014423012,"POLYGON ((-121.96678 37.5303, -121.96678 37.53...",1173.0,102617.0,1326900.0
1,060014060001,"POLYGON ((-122.26838 37.78803, -122.26736 37.7...",2035.0,65425.0,784200.0
2,060014337001,"POLYGON ((-122.11467 37.6887, -122.11462 37.68...",1508.0,91875.0,786400.0
3,060014011004,"POLYGON ((-122.26764 37.82783, -122.2676 37.82...",2675.0,98570.0,781300.0
4,060014012001,"POLYGON ((-122.26016 37.83119, -122.26003 37.8...",1443.0,235109.0,1344400.0


In [47]:
# =====================================================
# DENSITY CALCULATION
# =====================================================

gdf = gdf.to_crs(3310)

gdf["area_sqkm"] = (
    gdf.geometry.area / 1_000_000
)

gdf["pop_density"] = (
    gdf["population"] /
    gdf["area_sqkm"]
)

gdf = gdf.to_crs(4326)

In [48]:
# =====================================================
# FILTER TO OAKLAND (whole block groups, by % overlap)
# =====================================================

oakland = gpd.read_file("../../data/geo/OaklandCityLimits/OaklandCityLimits.shp").to_crs(gdf.crs)

# Calculate what % of each block group falls within Oakland
gdf["bg_area"] = gdf.geometry.area
intersection = gpd.overlay(gdf, oakland[["geometry"]], how="intersection")
intersection["overlap_pct"] = intersection.geometry.area / intersection["bg_area"]

# Keep block groups with >50% overlap (adjust threshold as needed)
inside = intersection[intersection["overlap_pct"] > .5]["GEOID"]
gdf = gdf[gdf["GEOID"].isin(inside)].drop(columns="bg_area")

/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_5253/686383270.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["bg_area"] = gdf.geometry.area
/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_5253/686383270.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  intersection["overlap_pct"] = intersection.geometry.area / intersection["bg_area"]


In [49]:
# =====================================================
# SAVE
# =====================================================

gdf.to_file(
    "../../data/geo/block_groups/oakland_BG.geojson",
    driver="GeoJSON"
)

print(
    f"Saved {len(gdf)} Alameda County block groups"
)

Saved 354 Alameda County block groups
